<a href="https://colab.research.google.com/github/kenleefk-edu/C3669C-2026-05/blob/main/WHISPER_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Google Colab Jupyter Notebooks to accomplish raw to polished **transcription** from a link with
* **Option A** (downloading and transcribing audio/video from links) and  
* **Option B** (performing structural text adjustments with zero paraphrasing).  

To achieve Option B without rewriting, restructuring, summarizing, or paraphrasing the speaker's words, the best workflow is to use OpenAI's Whisper library for the transcription, and an LLM (like Gemini or OpenAI via API) driven by a strict, rule-enforced system prompt for the structural cleanup.  

The complete code blocks below can be pasted into separate cells inside a single Google Colab notebook. Make sure to switch your notebook to a T4 GPU first (Runtime > Change runtime type > T4 GPU) to run the transcriber fast.  

### Step 1: Install Required Libraries
Run this cell first to download the tools for handling YouTube/audio URLs and the audio parser.  # cell ID

In [1]:
# cellID  RqkiWPfBhKxB
!pip install git+https://github.com/openai/whisper.git
!pip install yt-dlp
!pip install google-genai
!sudo apt-get install -y nodejs

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-axb14jtb
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-axb14jtb
  Resolved https://github.com/openai/whisper.git to commit 04f449b8a437f1bbd3dba5c9f826aca972e7709a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=69250bfbb9351d7c699cb157a14d4c4b2bde6df3159ce996d5ab1e7a6ce427b2
  Stored in directory: /tmp/pip-ephem-wheel-cache-hgp3n_mp/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.9 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state 

### Step 2. Mount Google Drive  ###

In [2]:
# cellID  XRf2LKMhpOQk
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CONFIGURATION: Change 'FOLDER_NAME' to your actual Drive folder  
- a folder after drive.mount volume e.g. "content/drive"

In [3]:
# cellID  dWMXC0xOkAXN
FOLDER_NAME = "/content/drive/MyDrive/SDGAI/C3659C/"
SOURCE_DIR = f"/content/drive/MyDrive/SDGAI/C3659C/"

### Option A: Transcribe Video or Audio from a Link  
This cell uses yt-dlp to extract the clean audio stream from almost any video/podcast URL (YouTube, Vimeo, Soundcloud, etc.) and transcribes it using Whisper.  

In [ ]:
import os
import whisper
import glob # Import glob to find files
from google.colab import drive

# Check if Google Drive is mounted, and mount if not
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# 1. INPUT: Define the directory where your .m4a audio files are located.
# This now uses the SOURCE_DIR from the previous cell.
AUDIO_FILE_DIRECTORY = SOURCE_DIR # Use the SOURCE_DIR defined in the previous cell

# Find all .m4a files in the specified directory
audio_files = glob.glob(os.path.join(AUDIO_FILE_DIRECTORY, "*.m4a"))

if not audio_files:
    print(f"No .m4a files found in '{AUDIO_FILE_DIRECTORY}'. Please make sure your audio files are in this directory.")
else:
    print(f"Found {len(audio_files)} .m4a files for transcription:")
    for audio_file_path in audio_files:
        print(f"- {os.path.basename(audio_file_path)}")

    print("\nLoading Whisper Model (Medium)...")
    # The Whisper model comes in various sizes, each offering a trade-off between speed, accuracy, and memory usage.
    # Choose the model that best fits your needs based on the audio length, desired accuracy, and available resources.

    # 'tiny': Fastest, lowest memory, but least accurate. Good for quick tests or very short audio.
    # model = whisper.load_model("tiny")

    # 'base': Faster than small, good balance for shorter audio or less critical accuracy needs.
    # model = whisper.load_model("base")

    # 'small': A good balance between speed and accuracy for most common use cases, especially for English speech.
    # model = whisper.load_model("small")

    # 'medium': Offers better accuracy than 'small' but is slower and uses more memory. Recommended for general purpose, higher quality transcription.
    model = whisper.load_model("medium") # Currently active as per your request

    # 'large': Highest accuracy but significantly slower and more memory-intensive. Best for critical applications or challenging audio.
    # model = whisper.load_model("large")

    # 'large-v2' or 'large-v3': Improved versions of 'large' with even better accuracy, often recommended for state-of-the-art results.
    # model = whisper.load_model("large-v2")
    # model = whisper.load_model("large-v3")

    # For English-only transcription, you can use '.en' versions (e.g., 'tiny.en', 'base.en', 'small.en', 'medium.en')
    # These are generally faster and more memory-efficient for English audio than their multilingual counterparts.
    # model = whisper.load_model("medium.en")

    for audio_file_path in audio_files:
        base_name = os.path.splitext(os.path.basename(audio_file_path))[0]
        # Construct the full path for the raw transcript file within the AUDIO_FILE_DIRECTORY
        raw_transcript_filename = os.path.join(AUDIO_FILE_DIRECTORY, f"{base_name}.txt")

        print(f"\nTranscribing '{os.path.basename(audio_file_path)}'... (This may take a few minutes depending on length)")
        try:
            result = model.transcribe(audio_file_path)
            raw_transcript = result["text"]

            # Save raw text to file with original filename in the AUDIO_FILE_DIRECTORY
            with open(raw_transcript_filename, "w", encoding="utf-8") as f:
                f.write(raw_transcript)

            print(f"--- RAW TRANSCRIPT GENERATED: '{raw_transcript_filename}' ---")
            print(raw_transcript[:500] + "... (truncated visual preview)")
        except Exception as e:
            print(f"Error transcribing {audio_file_path}: {e}")

    print("\nAll available .m4a files processed.")

Found 5 .m4a files for transcription:
- 11.15 Sufficient Modelling.m4a
- 15.15 Assumed Density Filtering.m4a
- 14.15 Kalman Filter Review.m4a
- 12.15 Prediction, Update, Likelihood.m4a
- 13.15 Estimators and Performance Evaluation.m4a

Loading Whisper Model (Medium)...

Transcribing '11.15 Sufficient Modelling.m4a'... (This may take a few minutes depending on length)
--- RAW TRANSCRIPT GENERATED: '/content/drive/MyDrive/SDGAI/C3659C/11.15 Sufficient Modelling.txt' ---
 There are many different ways that we can model the motion and model the measurement. And it's important to know that in multiple object tracking, these models don't need to be perfect. The models should instead be sufficiently accurate and also have a reasonable complexity. So what this means is that we would like to have high quality object estimates, but we want to obtain these estimates at a reasonable computational cost. So we need to find a balance between tracking accuracy and model comp... (truncated visual previ

### Option B: Rule-Enforced Clean Edit (No Paraphrasing/Summarizing)  
Standard regex text-replacement fails at adding sentence-based paragraphs or punctuation. An LLM is required, but it must be heavily constrained by rules.  

In [4]:
# cellID hJIc_upQiTaK
from google import genai
from google.genai import types
from google.colab import userdata
import glob # Import glob to find files
import os # Import os for path manipulation

# 1. Paste your free Gemini API key here
# API_KEY = "AIzaSy..."
API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

# Define the directory where your .txt files are located (same as audio files)
# This uses SOURCE_DIR from a previous cell.
TRANSCRIPT_FILE_DIRECTORY = SOURCE_DIR

# Find all raw transcript files generated by the previous step within the specified directory
raw_transcript_files = glob.glob(os.path.join(TRANSCRIPT_FILE_DIRECTORY, "*.txt"))

if not raw_transcript_files:
    print("No raw transcript (*.txt) files found to clean. Please run the transcription step first.")
else:
    print(f"Found {len(raw_transcript_files)} raw transcript files for cleaning:")
    for raw_transcript_file_path in raw_transcript_files:
        base_name = os.path.splitext(os.path.basename(raw_transcript_file_path))[0]
        # Construct the full path for the cleaned transcript file within the TRANSCRIPT_FILE_DIRECTORY
        clean_transcript_filename = os.path.join(TRANSCRIPT_FILE_DIRECTORY, f"{base_name}.md")

        print(f"\nCleaning and formatting '{os.path.basename(raw_transcript_file_path)}' structure...")

        # Load the raw transcript generated from the previous cell
        with open(raw_transcript_file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        # 2. Set up the strict system rules to prevent summarizing and paraphrasing
        system_instruction = """
You are a precision transcription editor. Your sole job is to do a limited light format edit.
STRICT LAWS YOU MUST FOLLOW:
1. Remove all verbal filler words and disfluencies (e.g., "um", "uh", "ah", "like", "you know", "so yeah").
2. Keep the speaker's exact words and original sentence structures intact.
3. Add missing punctuation (commas, periods, question marks) when necessary to make it readable.
4. Form logical paragraphs based on topic shifts, grouping together sentences.
5. CRITICAL: Do NOT paraphrase, rewrite, reorder, or edit the speaker's diction.
6. CRITICAL: Do NOT summarize, compress, or truncate the text. Every statement must remain.
"""

        response = client.models.generate_content(
            model='gemini-flash-latest', # Updated model to a potentially more available version
            # model='gemini-3.5-flash', # Previous model, kept as comment for reference
            contents=f"Please edit this transcript following the instructions:\n\n{raw_text}",
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.0 # Forced to 0.0 to prevent any creative rewriting/paraphrasing
            )
        )

        clean_transcript = response.text

        # Save the final structured Markdown file in the specified directory
        with open(clean_transcript_filename, "w", encoding="utf-8") as f:
            f.write(clean_transcript)

        print(f"--- FINAL MD TRANSCRIPT READY: '{clean_transcript_filename}' ---")
        print(f"Check your file explorer panel for '{clean_transcript_filename}'")

    print("\nAll raw transcript files cleaned.")

Found 5 raw transcript files for cleaning:

Cleaning and formatting '11.15 Sufficient Modelling.txt' structure...
--- FINAL MD TRANSCRIPT READY: '/content/drive/MyDrive/SDGAI/C3659C/11.15 Sufficient Modelling.md' ---
Check your file explorer panel for '/content/drive/MyDrive/SDGAI/C3659C/11.15 Sufficient Modelling.md'

Cleaning and formatting '15.15 Assumed Density Filtering.txt' structure...
--- FINAL MD TRANSCRIPT READY: '/content/drive/MyDrive/SDGAI/C3659C/15.15 Assumed Density Filtering.md' ---
Check your file explorer panel for '/content/drive/MyDrive/SDGAI/C3659C/15.15 Assumed Density Filtering.md'

Cleaning and formatting '14.15 Kalman Filter Review.txt' structure...
--- FINAL MD TRANSCRIPT READY: '/content/drive/MyDrive/SDGAI/C3659C/14.15 Kalman Filter Review.md' ---
Check your file explorer panel for '/content/drive/MyDrive/SDGAI/C3659C/14.15 Kalman Filter Review.md'

Cleaning and formatting '12.15 Prediction, Update, Likelihood.txt' structure...
--- FINAL MD TRANSCRIPT READY:

In [ ]:
# cellID  c428a2ca
import google.generativeai as genai
from google.colab import userdata

API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)

print("Listing available Gemini models and their supported methods:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"- {m.name} (supported for generateContent)")
    else:
        print(f"- {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Listing available Gemini models and their supported methods:
- models/gemini-2.5-flash (supported for generateContent)
- models/gemini-2.5-pro (supported for generateContent)
- models/gemini-2.0-flash (supported for generateContent)
- models/gemini-2.0-flash-001 (supported for generateContent)
- models/gemini-2.0-flash-lite-001 (supported for generateContent)
- models/gemini-2.0-flash-lite (supported for generateContent)
- models/gemini-2.5-flash-preview-tts (supported for generateContent)
- models/gemini-2.5-pro-preview-tts (supported for generateContent)
- models/gemma-4-26b-a4b-it (supported for generateContent)
- models/gemma-4-31b-it (supported for generateContent)
- models/gemini-flash-latest (supported for generateContent)
- models/gemini-flash-lite-latest (supported for generateContent)
- models/gemini-pro-latest (supported for generateContent)
- models/gemini-2.5-flash-lite (supported for generateContent)
- models/gemini-2.5-flash-image (supported for generateContent)
- models

### Why this approach works for your rules:  

-  **Whisper Engine:** Captures spoken syntax well but natively logs every "um" and "uh".  

-  **Temperature 0.0 Setting:** Setting the LLM temperature to 0.0 strips away its creativity. It treats the prompt as a strict engineering task, preventing it from inventing sentences or summarizing the core context.  

-  **Topic Chunking:** The AI acts as a smart structural map, adding logical double-returns (\n\n) to generate Markdown paragraphs when it identifies a shift in the speaker's main theme.  

Are you running into any issues with speaker diarization (identifying who is talking if there are multiple guests on the podcast), or would you like to add a feature that automatically timestamps the top of each paragraph?  